In [55]:
# code for processing behavioral data from SPCD
# read data from beh_raw folder and save to beh_processed for each subject
# we deal with these beh data:
# odor prediction, choice before and after meal

In [100]:
import os
import pandas as pd
from pathlib import Path
from openpyxl import Workbook

In [113]:
# common function across tasks used to process files
def process_file(subject_dir, pattern, expected_rows, cols_to_numeric, resp_col, rt_col,
                 correct_col, select_cols, run_col='Run', extra_mutate=None):
    files = list(subject_dir.glob(pattern))
    if len(files) != 1:
        raise ValueError(f"{pattern} not unique. Execution halted.")
    df = pd.read_csv(files[0])
    df[run_col] = pd.to_numeric(df[run_col], errors='coerce')
    df = df[df[run_col] > 0]
    if isinstance(expected_rows, list):
        if df.shape[0] not in expected_rows:
            raise ValueError(f"Unexpected number of rows: {df.shape[0]}")
    else:
        if df.shape[0] != expected_rows:
            raise ValueError(f"Expected {expected_rows} rows, got {df.shape[0]}")
    df[resp_col] = df[resp_col].astype(str).str.replace("'", "")
    df[cols_to_numeric] = df[cols_to_numeric].apply(pd.to_numeric, errors='coerce')
    df['resp'] = df[resp_col]
    df['rt'] = df[rt_col]
    df['acc'] = (df[correct_col] == df[resp_col]).astype(int)
    if extra_mutate:
        df = extra_mutate(df)
    return df[select_cols + ['resp', 'acc', 'rt']]

In [58]:
def load_pre_meal_probe(subject_dir):
    return process_file(
        subject_dir,
        pattern='Sub*_Day2_Probe_pre_*Trials.csv',
        expected_rows=72,
        cols_to_numeric=['trial', 'choice_class', 'set', 'choice_pair', 'cue_left_idx',
                         'cue_right_idx', 'flip', 'odor_left', 'odor_right', 'ITIDur',
                         'ISIDur', 'correct', 'key_resp.keys_raw', 'key_resp.rt_raw'],
        resp_col='key_resp.keys_raw',
        rt_col='key_resp.rt_raw',
        correct_col='correct',
        select_cols=['Run', 'trial', 'choice_class', 'set', 'choice_pair', 'cue_left_idx',
                     'cue_right_idx', 'flip', 'odor_left', 'odor_right', 'ITIDur', 'ISIDur', 'correct'],
        extra_mutate=lambda df: df.assign(choice_class=df['choice_class'].astype(str).map(mapping))
    )


In [59]:
def load_post_meal_probe(subject_dir):
    return process_file(
        subject_dir,
        pattern='Sub*_Day2_Probe_post_*Trials.csv',
        expected_rows=72,
        cols_to_numeric=['trial', 'choice_class', 'set', 'choice_pair', 'cue_left_idx',
                         'cue_right_idx', 'flip', 'odor_left', 'odor_right', 'ITIDur',
                         'ISIDur', 'correct', 'key_resp.keys_raw', 'key_resp.rt_raw'],
        resp_col='key_resp.keys_raw',
        rt_col='key_resp.rt_raw',
        correct_col='correct',
        select_cols=['Run', 'trial', 'choice_class', 'set', 'choice_pair', 'cue_left_idx',
                     'cue_right_idx', 'flip', 'odor_left', 'odor_right', 'ITIDur', 'ISIDur', 'correct'],
        extra_mutate=lambda df: df.assign(choice_class=df['choice_class'].astype(str).map(mapping))
    )

In [60]:
def load_odor_prediction(subject_dir):
    return process_file(
        subject_dir,
        pattern='Sub*_Day2_OdorPred_*PredTrials.csv',
        expected_rows=108,
        cols_to_numeric=['trial', 'cue_idx', 'odor_num', 'correct', 'ITIDur',
                         'odor_name_left', 'odor_name_center', 'odor_name_right',
                         'key_resp.keys_raw', 'key_resp.rt_raw'],
        resp_col='key_resp.keys_raw',
        rt_col='key_resp.rt_raw',
        correct_col='correct',
        select_cols=['Run', 'trial', 'cue_idx', 'odor_num', 'correct', 'ITIDur',
                     'odor_name_left', 'odor_name_center', 'odor_name_right']
    )

In [114]:
def load_preconditioning(subject_dir, day):
    pattern = f'Sub*_Day{day}_Preconditioning_*Trials.csv'
    
    if day == 1:
        expected_rows = [24 * n for n in [3, 4, 5]]
    elif day == 2:
        expected_rows = [24 * n for n in [1, 2]]
    else:
        raise ValueError(f"Unexpected day: {day}")

    df = process_file(
        subject_dir,
        pattern=pattern,
        expected_rows=expected_rows,
        cols_to_numeric=['trial', 'pair', 'cue1_idx', 'cue2_idx', 'left_idx', 'right_idx', 'ITIDur'],
        resp_col='key_resp.keys_raw',
        rt_col='key_resp.rt_raw',
        correct_col='correct',
        select_cols=['run', 'trial', 'pair', 'cue1_idx', 'cue2_idx', 'left_idx', 'right_idx', 'correct', 'ITIDur'],
        run_col='run'
    )
    return df


In [62]:
def load_recognition(subject_dir):
    return process_file(
        subject_dir,
        pattern='Sub*_Day2_Recognition_*Trials.csv',
        expected_rows=36,
        cols_to_numeric=['pair'],
        resp_col='key_resp.keys_raw',
        rt_col='key_resp.rt_raw',
        correct_col='correct',
        select_cols=['trial', 'pair', 'rep', 'cue1_idx', 'cue2_idx', 'correct', 'ITIDur']
    )

In [63]:
def load_character_task(subject_dir):
    return process_file(
        subject_dir,
        pattern='Sub*_Day1_IndepScan_*CharTrials.csv',
        expected_rows=128,
        cols_to_numeric=['rep', 'cue_idx', 'ITIDur', 'key_resp.rt_raw'],
        resp_col='key_resp.keys_raw',
        rt_col='key_resp.rt_raw',
        correct_col='correct',
        select_cols=['run', 'trial', 'rep', 'cue_idx', 'correct', 'ITIDur'],
        extra_mutate=lambda df: df.assign(run=pd.Series([1,2,3,4]).repeat(32).reset_index(drop=True))
    )

In [64]:
def load_place_task(subject_dir):
    return process_file(
        subject_dir,
        pattern='Sub*_Day1_IndepScan_*PlaceTrials.csv',
        expected_rows=128,
        cols_to_numeric=['rep', 'cue_idx', 'ITIDur', 'key_resp_2.rt_raw'],
        resp_col='key_resp_2.keys_raw',
        rt_col='key_resp_2.rt_raw',
        correct_col='correct',
        select_cols=['run', 'trial', 'rep', 'cue_idx', 'correct', 'ITIDur'],
        extra_mutate=lambda df: df.assign(run=pd.Series([1,2,3,4]).repeat(32).reset_index(drop=True))
    )

In [102]:
def save_dataframes(outdir, df_dict):
    for name, df in df_dict.items():
        df.to_csv(outdir / f"{name}.csv", index=False)

In [103]:
# Mapping for choice_class values
mapping = {
    '1': "B1>B3", '2': "B2>B3", '3': "B1>B2",
    '4': "A1>A3", '5': "A2>A3", '6': "A1>A2"
}

# Base data directory
datDir = Path('/Users/liuq13/SPCD_data/beh_raw')
outBaseDir = Path('/Users/liuq13/SPCD_data/beh_processed')

## load odor prediction task, pre-meal and post-meal probe tasks

In [12]:
for j in range(1, 43):
        SubID = f'sub-{j:02}' 
        subject_dir = datDir / SubID
        if not subject_dir.exists():
            continue

        print(f'Processing: {SubID}')
        outdir = outBaseDir / SubID
        outdir.mkdir(parents=True, exist_ok=True)

        try:
            pre_meal_dat = load_pre_meal_probe(subject_dir)
            post_meal_dat = load_post_meal_probe(subject_dir)
            odor_pred_dat = load_odor_prediction(subject_dir)

            save_dataframes(outdir, {
                'pre_meal_dat': pre_meal_dat,
                'post_meal_dat': post_meal_dat,
                'odor_pred_dat': odor_pred_dat
            })

        except Exception as e:
            print(f"❌ Error for {SubID}: {e}")
            continue

Processing: sub-01
Processing: sub-02
Processing: sub-03
Processing: sub-04
Processing: sub-07
Processing: sub-09
Processing: sub-10
Processing: sub-11
Processing: sub-13
Processing: sub-14
Processing: sub-15
Processing: sub-16
Processing: sub-17
Processing: sub-19
Processing: sub-20
Processing: sub-22
Processing: sub-23
Processing: sub-25
Processing: sub-26
Processing: sub-27
Processing: sub-28
Processing: sub-29
Processing: sub-30
Processing: sub-31
Processing: sub-32
Processing: sub-33
Processing: sub-34
Processing: sub-35
Processing: sub-36
Processing: sub-37
Processing: sub-38
Processing: sub-39
Processing: sub-40
Processing: sub-41


## load pre-conditioning data

In [115]:
for j in range(1, 43):
        SubID = f'sub-{j:02}' 
        subject_dir = datDir / SubID
        if not subject_dir.exists():
            continue

        print(f'Processing: {SubID}')
        outdir = outBaseDir / SubID
        outdir.mkdir(parents=True, exist_ok=True)

        for day in [1, 2]:
            try:
                pc_dat = load_preconditioning(subject_dir, day=day)
                save_dataframes(outdir, {
                    f'preconditioning_day{day}_dat': pc_dat
                })

            except Exception as e:
                print(f"❌ Error for {SubID} Day {day}: {e}")
                continue

Processing: sub-01
Processing: sub-02
Processing: sub-03
Processing: sub-04
Processing: sub-07
Processing: sub-09
Processing: sub-10
Processing: sub-11
Processing: sub-13
Processing: sub-14
Processing: sub-15
Processing: sub-16
Processing: sub-17
Processing: sub-19
Processing: sub-20
Processing: sub-22
Processing: sub-23
Processing: sub-25
Processing: sub-26
Processing: sub-27
Processing: sub-28
Processing: sub-29
Processing: sub-30
Processing: sub-31
Processing: sub-32
Processing: sub-33
Processing: sub-34
Processing: sub-35
Processing: sub-36
Processing: sub-37
Processing: sub-38
Processing: sub-39
Processing: sub-40
Processing: sub-41
